# Retrieval and Optimization
### Algorithm: Reciprocal Rank Fusion (RRF)

In [1]:
# Scenario: User asks "What is the architecture of BERT?"

# List 1: Dense Retrieval Results (Semantic Match)
# Vectors found papers discussing "Transformers" and "Encoders", 
# even if they don't explicitly say "BERT" in the chunk.
dense_results = [
    {"doc_id": "paper_A", "rank": 1, "content": "The Transformer relies entirely on self-attention..."},
    {"doc_id": "paper_B", "rank": 2, "content": "We propose a new Encoder stack for language understanding..."}, 
    {"doc_id": "paper_C", "rank": 3, "content": "BERT stands for Bidirectional Encoder Representations..."} 
]

# List 2: Sparse Retrieval Results (Keyword Match)
# BM25 found papers specifically containing the string "BERT".
# paper_C is #1 here because it contains the exact acronym.
# paper_D is irrelevant but has the word "Robert" which fuzzy matched or similar noise.
sparse_results = [
    {"doc_id": "paper_C", "rank": 1, "content": "BERT stands for Bidirectional Encoder Representations..."},
    {"doc_id": "paper_D", "rank": 2, "content": "My neighbor Robert has a similar architecture..."}, 
    {"doc_id": "paper_A", "rank": 3, "content": "The Transformer relies entirely on self-attention..."}
]

print("Data Loaded.")

Data Loaded.


In [ ]:
# Implement RRF Logic
from collections import defaultdict

def reciprocal_rank_fusion(results_list, k=60):
    """
    params:
    results_list: A list of lists. Each inner list contains retrieval results.
    k: The smoothing constant (60 optimal value indicate by Cormack et. (2009) is industry standard).
    """
    scores = defaultdict(float)
    
    # Iterate through every retriever's list
    for individual_list in results_list:
        # Iterate through the ranked documents
        for item in individual_list:
            doc_id = item['doc_id']
            rank = item['rank']
            
            # Apply Formula: 1 / (k + rank)
            # Note: The rank provided in data is 1-based. 
            # If using enumeration index, remember to add 1.
            score = 1 / (k + rank)
            
            scores[doc_id] += score
            
    # Sort by the summed RRF score (Highest first)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_scores

# Run RRF
final_ranking = reciprocal_rank_fusion([dense_results, sparse_results])

# Display 
print(f"{'Doc ID':<10} | {'RRF Score':<12} | {'Explanation'}")
print("-" * 50)
for doc_id, score in final_ranking:
    explanation = ""
    if doc_id == "paper_C": explanation = "Winner: Was #1 in Sparse, #3 in Dense (High Consensus)"
    if doc_id == "paper_A": explanation = "Runner Up: #1 in Dense, #3 in Sparse"
    if doc_id == "paper_B": explanation = "Low: Only appeared in Dense"
    
    print(f"{doc_id:<10} | {score:.6f}     | {explanation}")